# Sinus Scaled-Input Diagnostics

This notebook regenerates the scaled-input sensitivity diagnostics used in the thesis. It compares the standard sinus input range `[-1, 1]` with the scaled range `[-5, 5]`, while keeping the same target function and the same perturbation hyperparameters. The diagnostics are computed from frozen backpropagation checkpoints and averaged across 10 random seeds, all analysis mini-batches, and three checkpoints.

In [ ]:
from __future__ import annotations

import math
import os
import random
import shutil
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT_CANDIDATE = Path.cwd().resolve()
for candidate in [PROJECT_ROOT_CANDIDATE, *PROJECT_ROOT_CANDIDATE.parents]:
    if (candidate / "learning_rules_MLP.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find learning_rules_MLP.py. Run this notebook from the project folder or one of its subfolders.")

sys.path.insert(0, str(PROJECT_ROOT))

MPLCONFIGDIR = Path(tempfile.gettempdir()) / "sinus_scaled_input_matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

from learning_rules_MLP import (
    MLP,
    backprop_step,
    node_perturbation_step,
    node_perturbation_step_fan_in_scaled,
    node_perturbation_step_fixed_sigma,
    weight_perturb_step,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

OUTPUT_DIR = PROJECT_ROOT / "results_sinus_scaled_input"
FIGURE_DIR = OUTPUT_DIR / "figures"
DATA_DIR = OUTPUT_DIR / "data"
TABLE_DIR = OUTPUT_DIR / "tables"
for directory in [FIGURE_DIR, DATA_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size": 10,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
    }
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Output directory: {OUTPUT_DIR}")

## Configuration

The training setup matches the sinus diagnostic configuration used elsewhere in the thesis: the model is a sigmoid MLP with architecture `(8, 8, 8, 1)`, trained with backpropagation and frozen at epochs 1, 120, and 240. The perturbation sigmas are the final selected values for the sinus task. For each seed and checkpoint, diagnostics are averaged over the full set of non-overlapping analysis mini-batches from the training set.

In [ ]:
METHODS = ["np", "np_fan_in", "np_fixed", "wp"]
METHOD_LABELS = {
    "np": "IS-NP",
    "np_fan_in": "Fan-in NP",
    "np_fixed": "Vanilla NP",
    "wp": "WP",
}
METHOD_COLORS = {
    "np": "#ff7f0e",
    "np_fan_in": "#9467bd",
    "np_fixed": "#d62728",
    "wp": "#2ca02c",
}

DIMENSIONS = (8, 8, 8, 1)
TRAIN_BATCH_SIZE = 64
ANALYSIS_BATCH_SIZE = 64
NUM_PERTURBATION_ESTIMATES = 240
CHECKPOINT_EPOCHS = [1, 120, 240]
ANALYSIS_EPOCHS = max(CHECKPOINT_EPOCHS)
SEEDS = list(range(10))
BP_LR = 0.1

METHOD_SIGMAS = {
    "np": 0.15,
    "np_fan_in": 0.1,
    "np_fixed": 0.3,
    "wp": 0.225,
}

INPUT_CONDITIONS = [
    {"condition": "base", "input_scale": 1.0, "label": "[-1, 1]"},
    {"condition": "scaled", "input_scale": 5.0, "label": "[-5, 5]"},
]

## Data and Model Helpers

The target is generated from the unscaled latent input. The scaled condition therefore changes the input norm seen by the network without changing the underlying regression function. This isolates the effect of input scale on the perturbation estimators.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def synthetic_target(latent: torch.Tensor) -> torch.Tensor:
    input_dim = latent.shape[1]
    target = 0.6 * torch.sin(math.pi * latent).sum(dim=1, keepdim=True)
    target = target + 0.3 * latent.pow(2).sum(dim=1, keepdim=True)
    target = target + 0.2 * (latent[:, 0:1] * latent[:, 1:2])
    return target / math.sqrt(input_dim)


def load_sinus_data(
    input_scale: float,
    input_dim: int = 8,
    n_train: int = 512,
    n_test: int = 512,
    noise_std: float = 0.1,
    seed: int = 0,
):
    generator = torch.Generator().manual_seed(seed)

    latent_train = torch.rand(n_train, input_dim, generator=generator) * 2.0 - 1.0
    x_train = input_scale * latent_train
    y_train = synthetic_target(latent_train) + noise_std * torch.randn(n_train, 1, generator=generator)

    latent_test = torch.rand(n_test, input_dim, generator=generator) * 2.0 - 1.0
    x_test = input_scale * latent_test
    y_test = synthetic_target(latent_test)

    train_dataset = TensorDataset(x_train, y_train)
    test_dataset = TensorDataset(x_test, y_test)
    pin_memory = DEVICE.type == "cuda"

    return {
        "x_train": x_train,
        "y_train": y_train,
        "x_test": x_test,
        "y_test": y_test,
        "train_loader": DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, pin_memory=pin_memory),
        "train_eval_loader": DataLoader(train_dataset, batch_size=512, shuffle=False, pin_memory=pin_memory),
        "test_loader": DataLoader(test_dataset, batch_size=512, shuffle=False, pin_memory=pin_memory),
        "analysis_loader": DataLoader(
            train_dataset,
            batch_size=ANALYSIS_BATCH_SIZE,
            shuffle=False,
            drop_last=True,
            pin_memory=pin_memory,
        ),
    }


def make_model(require_grad: bool) -> MLP:
    return MLP(DIMENSIONS, activation=torch.sigmoid, require_grad=require_grad).to(DEVICE)


@torch.no_grad()
def evaluate_loss(model: MLP, loader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0
    total_examples = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        prediction = model(xb)
        loss_sum = F.mse_loss(prediction, yb, reduction="sum")
        total_loss += float(loss_sum)
        total_examples += xb.shape[0]
    return total_loss / max(total_examples, 1)

## Frozen Backpropagation Checkpoints

The diagnostics are computed at fixed parameter values. For each input condition, a backpropagation model is trained and saved at three checkpoints. The perturbation methods are then evaluated at those same frozen parameters.

In [ ]:
def train_backprop_checkpoints(data: dict, condition_label: str, seed: int) -> dict[int, dict[str, torch.Tensor]]:
    set_seed(seed)
    model = make_model(require_grad=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=BP_LR)
    checkpoint_states = {}

    for epoch in range(1, ANALYSIS_EPOCHS + 1):
        model.train()
        for xb, yb in data["train_loader"]:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            backprop_step(model, xb, yb, optimizer=optimizer)

        if epoch in CHECKPOINT_EPOCHS:
            checkpoint_states[epoch] = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }

        if epoch == 1 or epoch in CHECKPOINT_EPOCHS or epoch % 60 == 0:
            train_loss = evaluate_loss(model, data["train_eval_loader"])
            test_loss = evaluate_loss(model, data["test_loader"])
            print(
                f"seed {seed:2d} | {condition_label:>7} | epoch {epoch:3d}/{ANALYSIS_EPOCHS} | "
                f"train_loss={train_loss:.4f} | test_loss={test_loss:.4f}"
            )

    return checkpoint_states

## Diagnostic Computation

For every seed, checkpoint, and analysis mini-batch, the exact backpropagation update direction is computed first. Each perturbation method then draws `K = 240` independent perturbation-based gradient estimates at the same frozen parameters and on the same mini-batch. Cosine similarity is computed from the average of these estimates, while variance is computed as the coordinate-averaged squared deviation from the exact backpropagation update direction.

In [ ]:
def backprop_update_direction(model: MLP, xb: torch.Tensor, yb: torch.Tensor) -> torch.Tensor:
    optimizer = torch.optim.SGD(model.parameters(), lr=0.0)
    _, update = backprop_step(
        model,
        xb,
        yb,
        optimizer=optimizer,
        return_unscaled_parameter_update_vector=True,
    )
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model.zero_grad(set_to_none=True)
    return update.detach()


def perturbation_update_direction(method: str, model: MLP, xb: torch.Tensor, yb: torch.Tensor, sigma: float) -> torch.Tensor:
    step_functions = {
        "np": node_perturbation_step,
        "np_fan_in": node_perturbation_step_fan_in_scaled,
        "np_fixed": node_perturbation_step_fixed_sigma,
        "wp": weight_perturb_step,
    }
    with torch.no_grad():
        _, update = step_functions[method](
            model,
            xb,
            yb,
            eta=0.0,
            sigma=sigma,
            return_unscaled_parameter_update_vector=True,
        )
    return update.detach()


def cosine_similarity(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-12) -> float:
    denominator = torch.norm(a) * torch.norm(b)
    if float(denominator) < eps:
        return 0.0
    return float(torch.dot(a, b) / (denominator + eps))


def diagnostic_rows_for_condition(condition: dict, seed: int) -> pd.DataFrame:
    data = load_sinus_data(input_scale=condition["input_scale"], seed=seed)
    checkpoint_states = train_backprop_checkpoints(data, condition["label"], seed=seed)
    rows = []

    for checkpoint_epoch in CHECKPOINT_EPOCHS:
        model = make_model(require_grad=False)
        model.load_state_dict(checkpoint_states[checkpoint_epoch])
        for parameter in model.parameters():
            parameter.requires_grad_(False)

        print(f"\nDiagnostics | seed {seed:2d} | {condition['label']} | checkpoint {checkpoint_epoch}")
        batch_count = 0
        for batch_index, (xb, yb) in enumerate(data["analysis_loader"]):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            true_update = backprop_update_direction(model, xb, yb)

            for method in METHODS:
                sigma = METHOD_SIGMAS[method]
                estimates = [
                    perturbation_update_direction(method, model, xb, yb, sigma)
                    for _ in range(NUM_PERTURBATION_ESTIMATES)
                ]
                estimate_matrix = torch.stack(estimates, dim=0)
                mean_estimate = estimate_matrix.mean(dim=0)
                squared_errors = (estimate_matrix - true_update.unsqueeze(0)).pow(2)

                rows.append(
                    {
                        "seed": seed,
                        "condition": condition["condition"],
                        "condition_label": condition["label"],
                        "input_scale": condition["input_scale"],
                        "checkpoint_epoch": checkpoint_epoch,
                        "batch_index": batch_index,
                        "method": method,
                        "method_label": METHOD_LABELS[method],
                        "sigma": sigma,
                        "mean_estimate_cosine": cosine_similarity(mean_estimate, true_update),
                        "avg_sample_cosine": float(
                            np.mean([cosine_similarity(estimate, true_update) for estimate in estimate_matrix])
                        ),
                        "sample_variance": float(squared_errors.mean(dim=1).mean()),
                        "batch_variance": float((mean_estimate - true_update).pow(2).mean()),
                    }
                )
            batch_count += 1

        print(f"  analyzed {batch_count} batches")

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)

## Run Diagnostics

This cell performs the full scaled-input analysis and writes the raw batch-level data to CSV. It computes two input conditions, 10 seeds, three checkpoints, all analysis mini-batches, four perturbation methods, and 240 perturbation draws per method.

In [ ]:
all_batch_rows = []
for seed_index, seed in enumerate(SEEDS, start=1):
    print(f"\n===== Seed {seed_index}/{len(SEEDS)}: {seed} =====")
    for condition in INPUT_CONDITIONS:
        print(f"\n=== Condition {condition['label']} ===")
        all_batch_rows.append(diagnostic_rows_for_condition(condition, seed=seed))

batch_metrics_df = pd.concat(all_batch_rows, ignore_index=True)
batch_metrics_path = DATA_DIR / "sinus_scaled_input_batch_metrics.csv"
batch_metrics_df.to_csv(batch_metrics_path, index=False)
print(f"\nSaved batch-level diagnostics to {batch_metrics_path}")
display(batch_metrics_df.head())

## Summaries

The first aggregation averages over analysis mini-batches within each seed and checkpoint. Checkpoint summaries then average across seeds for each checkpoint. The overall summary first averages the three checkpoints within each seed, and then reports the mean and standard deviation across the 10 seeds.

In [ ]:
def summarize_diagnostics(batch_metrics_df: pd.DataFrame):
    checkpoint_seed = (
        batch_metrics_df.groupby(
            ["seed", "condition", "condition_label", "input_scale", "checkpoint_epoch", "method", "method_label"],
            as_index=False,
        )
        .agg(
            cosine=("mean_estimate_cosine", "mean"),
            variance=("sample_variance", "mean"),
            batch_variance=("batch_variance", "mean"),
            sigma=("sigma", "mean"),
            num_batches=("batch_index", "nunique"),
        )
        .sort_values(["seed", "condition", "checkpoint_epoch", "method"])
    )

    checkpoint_summary = (
        checkpoint_seed.groupby(
            ["condition", "condition_label", "input_scale", "checkpoint_epoch", "method", "method_label"],
            as_index=False,
        )
        .agg(
            cosine_similarity=("cosine", "mean"),
            cosine_similarity_std=("cosine", "std"),
            update_variance=("variance", "mean"),
            update_variance_std=("variance", "std"),
            batch_variance=("batch_variance", "mean"),
            sigma=("sigma", "mean"),
            num_seeds=("seed", "nunique"),
            num_batches=("num_batches", "mean"),
        )
        .sort_values(["condition", "checkpoint_epoch", "method"])
    )

    seed_over_checkpoints = (
        checkpoint_seed.groupby(
            ["seed", "condition", "condition_label", "input_scale", "method", "method_label"],
            as_index=False,
        )
        .agg(
            cosine=("cosine", "mean"),
            variance=("variance", "mean"),
            batch_variance=("batch_variance", "mean"),
            sigma=("sigma", "mean"),
        )
        .sort_values(["seed", "condition", "method"])
    )

    overall_summary = (
        seed_over_checkpoints.groupby(
            ["condition", "condition_label", "input_scale", "method", "method_label"],
            as_index=False,
        )
        .agg(
            cosine_similarity=("cosine", "mean"),
            cosine_similarity_std=("cosine", "std"),
            update_variance=("variance", "mean"),
            update_variance_std=("variance", "std"),
            batch_variance=("batch_variance", "mean"),
            sigma=("sigma", "mean"),
            num_seeds=("seed", "nunique"),
        )
        .sort_values(["condition", "method"])
    )

    return checkpoint_seed, checkpoint_summary, seed_over_checkpoints, overall_summary


checkpoint_seed_df, checkpoint_summary_df, seed_over_checkpoints_df, overall_summary_df = summarize_diagnostics(batch_metrics_df)
checkpoint_seed_path = DATA_DIR / "sinus_scaled_input_checkpoint_by_seed.csv"
checkpoint_summary_path = DATA_DIR / "sinus_scaled_input_checkpoint_summary.csv"
seed_over_checkpoints_path = DATA_DIR / "sinus_scaled_input_overall_by_seed.csv"
overall_summary_path = DATA_DIR / "sinus_scaled_input_overall_summary.csv"

checkpoint_seed_df.to_csv(checkpoint_seed_path, index=False)
checkpoint_summary_df.to_csv(checkpoint_summary_path, index=False)
seed_over_checkpoints_df.to_csv(seed_over_checkpoints_path, index=False)
overall_summary_df.to_csv(overall_summary_path, index=False)

print(f"Saved checkpoint-by-seed summary to {checkpoint_seed_path}")
print(f"Saved checkpoint summary to {checkpoint_summary_path}")
print(f"Saved overall-by-seed summary to {seed_over_checkpoints_path}")
print(f"Saved overall summary to {overall_summary_path}")
display(overall_summary_df)

## Plots

The main thesis figures average across all three checkpoints. The checkpoint-specific plots are generated for the appendix.

In [ ]:
def save_pdf(fig, filename: str) -> Path:
    path = FIGURE_DIR / filename
    fig.savefig(path, format="pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return path


def cosine_axis_limits(values, padding_fraction=0.15):
    values = np.asarray([float(value) for value in values if np.isfinite(float(value))])
    if values.size == 0:
        return 0.0, 1.0
    min_value = float(values.min())
    max_value = float(values.max())
    if min_value >= 0.0:
        upper = max_value * (1.0 + padding_fraction) if max_value > 0.0 else 0.05
        return 0.0, min(1.0, upper)
    value_range = max_value - min_value
    padding = value_range * padding_fraction if value_range > 0.0 else 0.05
    return max(-1.0, min_value - padding), min(1.0, max_value + padding)


def plot_scaled_comparison(summary_df: pd.DataFrame, metric: str, ylabel: str, filename: str, checkpoint_epoch=None):
    if checkpoint_epoch is not None:
        plot_df = summary_df[summary_df["checkpoint_epoch"] == checkpoint_epoch].copy()
    else:
        plot_df = summary_df.copy()

    base_df = plot_df[plot_df["condition"] == "base"].set_index("method")
    scaled_df = plot_df[plot_df["condition"] == "scaled"].set_index("method")
    methods = [method for method in METHODS if method in base_df.index and method in scaled_df.index]

    base_values = np.asarray([float(base_df.loc[method, metric]) for method in methods])
    scaled_values = np.asarray([float(scaled_df.loc[method, metric]) for method in methods])
    all_values = np.concatenate([base_values, scaled_values])

    fig, ax = plt.subplots(figsize=(5.8, 3.4))
    x = np.arange(len(methods))
    width = 0.34

    for index, method in enumerate(methods):
        color = METHOD_COLORS[method]
        ax.bar(
            x[index] - width / 2,
            base_values[index],
            width=width,
            color=color,
            edgecolor="black",
            linewidth=0.5,
        )
        ax.bar(
            x[index] + width / 2,
            scaled_values[index],
            width=width,
            color=color,
            edgecolor="black",
            linewidth=0.5,
            hatch="////",
            alpha=0.78,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([METHOD_LABELS[method] for method in methods], rotation=20, ha="right")
    ax.set_ylabel(ylabel)

    legend_handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor="0.6", edgecolor="black", label="[-1, 1]"),
        plt.Rectangle((0, 0), 1, 1, facecolor="0.6", edgecolor="black", hatch="////", alpha=0.78, label="[-5, 5]"),
    ]
    ax.legend(handles=legend_handles, frameon=True, framealpha=0.95)

    if metric == "update_variance":
        positive_values = all_values[all_values > 0.0]
        ax.set_yscale("log")
        ax.set_ylabel(f"{ylabel} (log scale)")
        ax.set_ylim(float(positive_values.min()) / 3.0, float(positive_values.max()) * 3.0)
        ax.grid(True, which="both", axis="y", alpha=0.25)
    elif metric == "cosine_similarity":
        ax.set_ylim(*cosine_axis_limits(all_values))

    fig.tight_layout()
    return save_pdf(fig, filename)


figure_paths = []
figure_paths.append(
    plot_scaled_comparison(
        overall_summary_df,
        metric="update_variance",
        ylabel="Update variance",
        filename="sinus_scaled_input_variance_comparison.pdf",
    )
)
figure_paths.append(
    plot_scaled_comparison(
        overall_summary_df,
        metric="cosine_similarity",
        ylabel="Cosine similarity",
        filename="sinus_scaled_input_cosine_comparison.pdf",
    )
)

for checkpoint_epoch in CHECKPOINT_EPOCHS:
    figure_paths.append(
        plot_scaled_comparison(
            checkpoint_summary_df,
            metric="update_variance",
            ylabel="Update variance",
            filename=f"sinus_scaled_input_variance_checkpoint_{checkpoint_epoch:03d}.pdf",
            checkpoint_epoch=checkpoint_epoch,
        )
    )
    figure_paths.append(
        plot_scaled_comparison(
            checkpoint_summary_df,
            metric="cosine_similarity",
            ylabel="Cosine similarity",
            filename=f"sinus_scaled_input_cosine_checkpoint_{checkpoint_epoch:03d}.pdf",
            checkpoint_epoch=checkpoint_epoch,
        )
    )

print("Wrote figures:")
for path in figure_paths:
    print(f"  {path}")

## Download Archive

The final cell creates a zip archive containing the generated PDFs and CSV files. In Colab, uncomment the download lines to download the archive directly.

In [ ]:
zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print(f"Wrote archive: {zip_path}")

# Uncomment in Colab for immediate download.
# from google.colab import files
# files.download(zip_path)